# Human Judge, Then LLM Judge

For subjective tasks — tone, humor, persuasiveness — string and embedding metrics fall short. The standard playbook is: first make *yourself* the metric, label ~20–50 examples by hand and capture your reasoning, then train an LLM judge on those labels and validate it agrees with you.

> **This notebook is interactive.** The `human_metric` cell calls Python's `input()` to ask for your judgement. Run it cell-by-cell in JupyterLab or VS Code — do **not** run it headlessly with `jupyter execute` or `papermill`, those will hang waiting for stdin.

In [ ]:
%pip install -r ../requirements.txt -q

In [ ]:
import os
import dspy
from dotenv import load_dotenv

load_dotenv()

dspy.configure(lm=dspy.LM('openai/gpt-5-mini'))

## A tiny example program and dataset

We need a program to evaluate and a small dataset of inputs. The exact task doesn't matter — what matters is that you have a `question` field as the input and the program emits an `answer` field.

In [ ]:
class AnswerQuestion(dspy.Signature):
    """Answer the user's question concisely and accurately."""
    question: str = dspy.InputField()
    answer: str = dspy.OutputField()


program = dspy.Predict(AnswerQuestion)

dataset = [
    dspy.Example(question="What is the capital of France?").with_inputs("question"),
    dspy.Example(question="Who wrote Pride and Prejudice?").with_inputs("question"),
    dspy.Example(question="What is the chemical symbol for gold?").with_inputs("question"),
    dspy.Example(question="What year did World War II end?").with_inputs("question"),
    dspy.Example(question="How many continents are there?").with_inputs("question"),
]

## Step 1 — Make yourself the metric

The `human_metric` below displays each prediction, asks you for a yes/no judgement, asks you *why*, and appends the result to `judge_dataset.jsonl`. The reasoning field is the most valuable thing you produce here — it's what teaches the future LLM judge how to think about your task. Set `num_threads=1` so prompts appear one at a time.

In [ ]:
import json

def human_metric(example, pred, trace=None):
    # 1. Display the interaction clearly
    print(f"\n{'='*40}")
    print(f"Input:  {example.question}")
    print(f"Output: {pred.answer}")
    print(f"{'-'*40}")

    # 2. Ask for judgment (block until valid input)
    while True:
        vote = input("Is this correct? (y/n) > ").lower().strip()
        if vote in ['y', 'n']:
            break

    # 3. Capture reasoning (critical for training future judges!)
    # Don't skip this! 'Why' is more valuable than 'Yes/No' for LLMs.
    reasoning = input("Reasoning (Why?):      > ").strip()

    score = 1.0 if vote == 'y' else 0.0

    # 4. Save the result as a side-effect
    # This builds your training set automatically while you evaluate.
    with open("judge_dataset.jsonl", "a") as f:
        record = {
            "question": example.question,
            "answer": pred.answer,
            "is_good": score == 1.0,
            "reasoning": reasoning,
        }
        f.write(json.dumps(record) + "\n")

    return score


# Run the evaluation
# CRITICAL: Set num_threads=1 so questions appear one at a time!
evaluate_with_human = dspy.Evaluate(
    devset=dataset,  # Start with just a few examples
    metric=human_metric,
    num_threads=1,
    display_progress=True,
    display_table=0,
)

accuracy = evaluate_with_human(program)
print(f"Accuracy: {accuracy}")

## Step 2 — Load your hand-labeled data into DSPy Examples

Once you've labeled at least 20–50 examples (more if your task is subtle), convert the JSONL into DSPy `Example` objects. The judge takes `question` and `answer` as inputs and predicts `is_good` plus a `reasoning` explanation.

In [ ]:
# Load the data you just created
judge_trainset = []

with open("judge_dataset.jsonl", "r") as f:
    for line in f:
        record = json.loads(line)
        judge_trainset.append(
            dspy.Example(
                question=record["question"],
                answer=record["answer"],
                is_good=record["is_good"],      # The label (True/False)
                reasoning=record["reasoning"],  # The explanation (Chain-of-Thought)
            ).with_inputs("question", "answer")
        )

print(f"Loaded {len(judge_trainset)} examples")

## Step 3 — Define the judge signature

The judge sees the question, the candidate answer, and a gold answer for reference, then predicts whether the candidate is good and explains its reasoning.

In [ ]:
class JudgeQuality(dspy.Signature):
    """Evaluate whether a model's answer is accurate and helpful."""
    question: str = dspy.InputField()
    answer: str = dspy.InputField()
    gold_answer: str = dspy.InputField(desc="The correct answer for reference")
    is_good: bool = dspy.OutputField(
        desc="True if answer is accurate and helpful, False otherwise"
    )
    reasoning: str = dspy.OutputField(desc="Explain your judgment")

## Step 4 — Train the judge with BootstrapFewShot

We define a metric that checks whether the judge agrees with the human label, then use `BootstrapFewShot` to seed the judge with few-shot examples that match our judgements. BootstrapFewShot is a lightweight optimizer — it just selects examples that pass the metric and inserts them into the prompt.

In [ ]:
def judge_metric(example, pred, trace=None):
    return pred.is_good == example.is_good


# Start with baseline
judge = dspy.ChainOfThought(JudgeQuality)

# Optimize
optimizer = dspy.BootstrapFewShot(
    metric=judge_metric,
    max_bootstrapped_demos=8,
    max_labeled_demos=8,
)

# Split your labels — train on most, hold out a few for validation
split = max(1, int(len(judge_trainset) * 0.8))
train_split = judge_trainset[:split]
val_split = judge_trainset[split:]

optimized_judge = optimizer.compile(
    student=judge,
    trainset=train_split,
)

## Step 5 — Validate judge accuracy

Check how well the optimized judge agrees with your held-out labels. If agreement is in the 80–90% range, the judge is trustworthy enough to use as the metric for optimizing your real program. Below that, collect more labels or refine the judge signature.

In [ ]:
from dspy.evaluate import Evaluate

# Set up the evaluator
evaluator = Evaluate(
    devset=val_split,
    metric=judge_metric,
    num_threads=4,
    display_progress=True,
    display_table=0,
)

# Run evaluation
accuracy = evaluator(optimized_judge)
print(f"Judge accuracy: {accuracy:.1%}")

Once the judge clears the agreement bar, swap it in as the metric for your main program. The next notebook (`rubric-and-multipredictor.ipynb`) shows how to extend the same pattern to multi-dimensional rubrics and per-predictor feedback for GEPA.